<a href="https://colab.research.google.com/github/JuanZapa7a/Medical-Image-Processing/blob/main/PIM_Challenge/PIM_Challenge_Student_Practica_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UPCT Medical Image Segmentation Challenge 2026-27
## Práctica 9

**Asignatura:** Procesado de Imágenes Médicas (521104007)

**Profesor:** Juan Zapata

> Contenido **nuevo** de esta práctica. Copia las celdas de aquí abajo y pégalas **al final** de tu propio notebook (el que empezaste en la Práctica 6) — no repitas las prácticas anteriores, ya las tienes hechas ahí.

## Guía de Sesiones (2 horas por sesión)
| Práctica | Fechas (Grupo A / B) | Objetivo de la Sesión | Checkpoint Visual |
|----------|----------------------|-----------------------|-------------------|
| **P6** | 28 Oct - 2 Nov | EDA, Dataset y formato RLE | 6 imágenes con máscaras + RLE OK |
| **P7** | 9-11 Nov | Baseline U-Net y 1ª Submission | Gráficas de Loss + Submission Kaggle |
| **P8** | 16-18 Nov | Data Augmentation y mejora | Comparativa Baseline vs Augmented |
| ▶ **P9** | 23-25 Nov | Inferencia, Threshold y Errores | 5 imágenes normales + 2 casos de error |
| **P10** | 30 Nov-2 Dic | TTA, Submission Final y Defensa | Mejor Dice Score + Defensa Oral |

> **Regla de Oro:** Según el Art. 7.5 del Reglamento de Evaluación UPCT, la asistencia y validación del Checkpoint en el aula es obligatoria para superar la práctica.


# Práctica 9: Inferencia, Optimización de Threshold y Análisis de Errores
## Sesión única (23 Nov Grupo A / 25 Nov Grupo B)

### Objetivos de la sesión:
1. Entender por qué el umbral de decisión (threshold) de 0.5 no siempre es el óptimo en medicina.
2. Calcular el **Threshold Óptimo** evaluando el conjunto de validación.
3. Realizar un **Análisis Cualitativo de Errores** (Falsos Positivos y Falsos Negativos).
4. Generar la **Submission Final** para Kaggle.

###  Contexto Clínico: El dilema FP vs FN
En la detección de cáncer de mama:
*   **Falso Negativo (FN):** Decir que no hay tumor cuando sí lo hay. (Grave: retraso en el tratamiento).
*   **Falso Positivo (FP):** Decir que hay tumor cuando no lo hay. (Ansiedad, biopsias innecesarias).

Nuestro modelo debe encontrar el equilibrio perfecto. Hoy vamos a buscarlo.

> **CHECKPOINT P9:** Mostrar al profesor:
> 1. Gráfica de Threshold vs Dice Score con el punto óptimo marcado.
> 2. Visualización de 5 casos críticos (incluyendo al menos 1 imagen "normal" y 1 error grave).

## Antes de empezar: elegir el modelo definitivo (P7 baseline vs P8 augmented)

En el Bloque 9.3 se habla del "modelo definitivo": el que mejor Val Dice haya conseguido entre el baseline (P7) y el augmented (P8). La celda siguiente carga **desde Drive** los checkpoints de ambos, compara su Val Dice, y deja el ganador en la variable `model` — así no hay que decidirlo a mano ni arriesgarse a usar por descuido el que no tocaba.


In [ ]:
# ============================================================
# Elegir el "modelo definitivo": el mejor Val Dice entre baseline y augmented
# ============================================================
paths = {
    'baseline (P7)': CHECKPOINT_DIR / 'baseline_checkpoint.pth',
    'augmented (P8)': CHECKPOINT_DIR / 'augmented_checkpoint.pth',
}

candidates = {}
for name, path in paths.items():
    if path.exists():
        ckpt = torch.load(path, map_location=DEVICE)
        candidates[name] = ckpt
        print(f"{name}: Val Dice = {ckpt['best_val_dice']:.4f}")
    else:
        print(f"{name}: no encontrado en Drive ({path})")

if not candidates:
    raise FileNotFoundError("No se encontró ningún checkpoint. Completa antes la P7 y/o la P8.")

best_name = max(candidates, key=lambda n: candidates[n]['best_val_dice'])
best_checkpoint = candidates[best_name]

model = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=1, activation=None)
model = model.to(DEVICE)
model.load_state_dict(best_checkpoint['model_state_dict'])

print(f"\nModelo definitivo: {best_name} (Val Dice = {best_checkpoint['best_val_dice']:.4f})")


## Bloque 9.1: El Threshold como Hiperparámetro
### 0.5 nunca fue una decisión informada

Desde la Práctica 7 habéis usado `probs > 0.5` para convertir probabilidades en máscara binaria. Ese 0.5 no salió de ningún análisis de vuestros datos: es simplemente el punto medio del rango de `sigmoid()` (que va de 0 a 1), la opción por defecto más "neutra" posible. Nada garantiza que sea el mejor corte para este dataset en concreto.

> **Idea clave:** el threshold es un hiperparámetro más de vuestro sistema, igual que `lr` o `NUM_EPOCHS` — simplemente hasta ahora lo habíais dejado fijo sin cuestionarlo.

### Por qué el threshold no es gratis: precisión vs recall

Recordad el dilema clínico de la introducción de esta práctica: un Falso Negativo (no detectar un tumor real) y un Falso Positivo (detectar un tumor que no existe) no tienen el mismo coste. El threshold controla directamente ese equilibrio:

| Threshold | Efecto en la máscara | Consecuencia |
|-----------|------------------------|----------------|
| Más bajo (ej. 0.3) | El modelo marca "tumor" con menos evidencia | Menos Falsos Negativos, más Falsos Positivos |
| Más alto (ej. 0.7) | El modelo exige más confianza para marcar "tumor" | Menos Falsos Positivos, más Falsos Negativos |

El Dice Score que vais a maximizar es una media que pondera precisión y recall por igual — no distingue si os equivocáis por FP o por FN. Maximizar Dice os da el mejor equilibrio *estadístico*, pero no necesariamente el mejor equilibrio *clínico* (donde un FN suele ser más grave que un FP).

> **Pregunta para pensar:** si fuerais los responsables clínicos de este sistema y tuvierais que elegir entre el threshold que maximiza el Dice y uno ligeramente distinto que reduce los Falsos Negativos a costa de algo de Dice, ¿cuál eligiríais? ¿Por qué el ejercicio, aun así, os pide maximizar el Dice?

### Por qué se busca en validación, no en train ni en test

Este es exactamente el mismo principio de las Prácticas 7 y 8 aplicado a un hiperparámetro nuevo:

| Conjunto | ¿Por qué NO usarlo para elegir el threshold? |
|----------|-------------------------------------------------|
| Train | El modelo ya vio estos datos al entrenar; el threshold quedaría ajustado al ruido específico de esas imágenes |
| Test | No tiene máscaras — no podéis calcular Dice ahí. Y aunque las tuvierais, elegir el threshold mirando el test sería "hacer trampa": estaríais ajustando un hiperparámetro con los mismos datos con los que luego os evalúan |
| Val | Datos que el modelo no usó para ajustar pesos, y sí tenéis máscaras para medir Dice — la elección justa |

### Por qué no hace falta reentrenar para probar cada threshold

A diferencia de cambiar `lr` o añadir augmentation (que obligan a entrenar el modelo entero otra vez), el threshold se aplica **después** de que el modelo ya generó sus probabilidades. Los pesos del modelo no cambian entre un threshold y otro — solo cambia dónde cortáis esas probabilidades para decidir 0 o 1. Por eso podéis barrer 17 valores de threshold (0.1 a 0.9 en pasos de 0.05) evaluando el `val_loader` una vez por cada uno, sin ningún coste de entrenamiento.

### Cómo interpretar la curva Threshold vs Dice

La forma típica de esta curva es una campana: Dice bajo en los extremos (thresholds muy bajos generan demasiados Falsos Positivos; muy altos, demasiados Falsos Negativos) y un máximo en algún punto intermedio.

| Forma de la curva | Interpretación |
|----------------------|-------------------|
| Pico marcado y estrecho | El threshold importa mucho; elegir mal cuesta Dice de forma notable |
| Curva plana en la zona central | El modelo es robusto al threshold exacto; varios valores cercanos dan un Dice similar |

### Resumen rápido

| Concepto | Idea principal |
|----------|-----------------|
| Threshold | Un hiperparámetro más, no una constante fija |
| Precisión vs recall | Threshold bajo → menos FN, más FP; threshold alto → lo contrario |
| Dónde buscarlo | En validación: train sesga el ajuste, test invalidaría la evaluación |
| Coste de la búsqueda | Ninguno de entrenamiento — el modelo no cambia, solo el corte de decisión |
| Forma de la curva | Un pico marcado indica que el threshold importa; una curva plana indica que el modelo es robusto en un rango de valores |

## Tarea 9.1: Búsqueda del Threshold Óptimo
Hasta ahora hemos usado `threshold = 0.5` para binarizar las probabilidades. Pero, ¿es realmente el mejor valor para nuestro dataset?

### Instrucciones:
1. Pon el modelo en modo evaluación (`model.eval()`).
2. Itera sobre el `val_loader` (sin calcular gradientes).
3. Para cada batch, obtén las probabilidades (`torch.sigmoid(logits)`).
4. Prueba un rango de thresholds desde **0.1 hasta 0.9** (con pasos de 0.05).
5. Para cada threshold, calcula el **Dice Score promedio** en todo el conjunto de validación.
6. Grafica: **Eje X** = Threshold, **Eje Y** = Dice Score.
7. Identifica y guarda el `best_threshold` que maximiza el Dice.

> **Pista:** No modifiques los pesos del modelo. Solo estás evaluando cómo cambia la métrica al cambiar el corte de decisión.

In [ ]:
# ============================================================
# TAREA 9.1: BÚSQUEDA DEL THRESHOLD ÓPTIMO
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

# ESCRIBE TU CÓDIGO AQUÍ
# 1. Define una lista de thresholds a probar: thresholds = np.arange(0.1, 0.95, 0.05)
# 2. Inicializa una lista para guardar los dices: val_dices_per_threshold = []
# 3. model.eval()
# 4. Bucle for threshold in thresholds:
#       epoch_dice = 0
#       with torch.no_grad():
#           for images, masks in val_loader:
#               images, masks = images.to(DEVICE), masks.to(DEVICE)
#               logits = model(images)
#               probs = torch.sigmoid(logits)
#               preds = (probs > threshold).float() # <-- Aquí usas el threshold del bucle
#               # Calcular Dice...
#               epoch_dice += dice.item()
#       val_dices_per_threshold.append(epoch_dice / len(val_loader))
# 5. Graficar y encontrar el mejor threshold.

# Tu código aquí...

print(f"Mejor Threshold encontrado: {best_threshold:.2f} con Dice: {max_dice:.4f}")

## Bloque 9.2: Leer los Errores de un Modelo de Segmentación
### Por qué una media no basta

Un Dice promedio de 0.80 en validación puede estar compuesto de muchas formas distintas: 100 imágenes con Dice 0.80 cada una, o 90 imágenes con Dice 0.95 y 10 imágenes con Dice 0.05. La media no distingue estos dos escenarios, pero clínicamente son radicalmente distintos — el segundo caso significa que hay un subgrupo de pacientes donde el modelo falla por completo, oculto detrás de un buen promedio general.

### TP, TN, FP y FN a nivel de píxel

Hasta ahora habéis calculado Dice e IoU como fórmulas sobre conjuntos de píxeles, sin nombrar explícitamente sus cuatro componentes. En segmentación, cada píxel de la máscara predicha cae en una de estas cuatro categorías al compararlo con el ground truth:

| | GT = tumor | GT = fondo |
|---|---|---|
| **Predicción = tumor** | TP (acierto) | FP (falsa alarma) |
| **Predicción = fondo** | FN (fallo, no lo detecta) | TN (acierto) |

En una imagen **normal** (máscara real completamente vacía, sin ningún píxel de tumor), es matemáticamente imposible tener TP o FN — no hay tumor que acertar ni que perder. Los únicos resultados posibles son TN (todo el fondo correctamente ignorado) o FP (el modelo "alucina" un tumor donde no existe). Por eso el Caso 1 de esta tarea (una imagen normal) es la comprobación más directa y limpia de si vuestro modelo genera Falsos Positivos.

### Por qué elegir los casos a propósito, y no al azar

El dataset BUSI está desbalanceado (437 benign, 210 malignant, 133 normal). Si eligierais 5 imágenes puramente al azar del `val_df`, lo más probable es que las 5 sean `benign` — la clase mayoritaria — y nunca lleguéis a ver cómo se comporta el modelo con imágenes `normal` o `malignant`. Es el mismo problema del split del Bloque 7.1, aplicado ahora a la inspección visual: un muestreo aleatorio ciego a las clases minoritarias no os enseña nada sobre ellas. Por eso la tarea pide explícitamente un caso de cada clase, y no "5 imágenes cualquiera".

### Cómo encontrar los "fallos estrepitosos" sin depender de la suerte

Para los Casos 4 y 5, la instrucción permite buscar "iterando" en vez de al azar. La forma sistemática de hacerlo es: calcular el Dice individual de cada imagen de `val_df` (no el promedio, uno por imagen),
y ordenar de menor a mayor. Las imágenes con el Dice más bajo son, por definición, aquelca más — encontrarlas por fuerza bruta es mucho más fiable que confiar en que os toquenpor azar en una muestra aleatoria pequeña.

### Qué mirar en el overlay

El overlay (predicción en rojo sobre la imagen original) es donde se ve el *tipo* de error, no solo si hubo error:

| Patrón visual | Tipo de error | Efecto en Dice |
|-----------------|-----------------|-------------------|
| Rojo se queda corto respecto al contorno real del tumor | Bajo-segmentación (FN en el borde) | Reduce Dice |
| Rojo se extiende más allá del contorno real | Sobre-segmentación (FP en el borde) | Reduce Dice |
| Rojo aparece sobre tejido sano sin tumor cerca | Falso Positivo "alucinado" | Reduce Dice y es clínicamente grave |
| No hay rojo donde sí hay tumor | Falso Negativo total | El error más grave clínicamente |

### De lo técnico a lo clínico

El checkpoint no pide solo identificar TP/TN/FP/FN — pide explicar **qué significa clínicamente** cada tipo de error. No es lo mismo decir "hay píxeles FP en el borde superior" que decir "el modelo marcaría una zona sana como sospechosa, lo que llevaría a pruebas o biopsias innecesarias en esta paciente". La segunda frase es la que conecta con el dilema FP/FN de la introducción de esta práctica.

### Resumen rápido

| Concepto | Idea principal |
|----------|-----------------|
| Media vs casos individuales | Un buen Dice promedio puede ocultar fallos graves en subgrupos concretos |
| TP/TN/FP/FN por píxel | En imágenes normales solo son posibles TN o FP — nunca TP ni FN |
| Selección deliberada | Muestreo aleatorio en datos desbalanceados esconde las clases minoritarias |
| Buscar fallos | Ordenar por Dice individual, no confiar en el azar |
| Overlay | El patrón visual (falta, sobra, o aparece donde no debería) indica el tipo de error |
| Explicación clínica | Traducir TP/FP/FN a consecuencias reales para la paciente |

## Tarea 9.2: Análisis Cualitativo de Errores
Las métricas numéricas no lo cuentan todo. Un Dice de 0.80 puede ocultar errores graves en casos clínicamente relevantes. Vamos a visualizar **dónde** está fallando el modelo.

### Instrucciones:
1. Selecciona manualmente (o aleatoriamente) 5 imágenes del `val_df` que representen:
   * **Caso 1:** Una imagen de clase **"normal"** (para ver si hay Falsos Positivos).
   * **Caso 2:** Una imagen de clase **"benign"** con tumor pequeño.
   * **Caso 3:** Una imagen de clase **"malignant"** con bordes irregulares.
   * **Caso 4 y 5:** Dos imágenes donde el modelo haya fallado estrepitosamente (puedes buscarlas iterando si quieres, o usar aleatorias).
2. Para cada una, genera una figura con 4 columnas:
   * **Original**
   * **Ground Truth (Máscara Real)**
   * **Predicción** (usando tu `best_threshold`)
   * **Overlay** (Predicción en rojo sobre la Original)
3. Añade un título a cada imagen indicando su clase y si es TP, TN, FP o FN.

> **CHECKPOINT P9.1:** Muestra al profesor las 5 imágenes y explica clínicamente qué está pasando en los errores.

In [ ]:
# ============================================================
# TAREA 9.2: ANÁLISIS CUALITATIVO DE ERRORES
# ============================================================
import cv2

# ESCRIBE TU CÓDIGO AQUÍ
# 1. Selecciona 5 índices de val_df (puedes usar val_df.sample(5) o elegirlos a dedo)
# 2. Crea una figura plt.subplots(5, 4, figsize=(16, 20))
# 3. Bucle para cada imagen:
#       - Carga la imagen y la máscara real
#       - Pasa la imagen por el modelo (preprocesando igual que en el Dataset)
#       - Calcula la predicción con best_threshold
#       - Muestra las 4 columnas (Original, GT, Pred, Overlay)

# Tu código aquí...

plt.tight_layout()
plt.show()

## Bloque 9.3: Cerrar el Círculo — De la Calibración a la Submission
### El mismo tipo de pipeline que ya construisteis en la Práctica 7

Esta tarea os pide repetir, conceptualmente, el mismo pipeline de inferencia del Bloque 7.4: cargar imagen de test, preprocesarla igual que en entrenamiento, pasarla por el modelo, umbralizar, redimensionar la máscara al tamaño original, convertir a RLE y guardar el CSV. Lo único que cambia es **qué valor de threshold usáis para binarizar**: antes era `0.5` fijo, ahora es vuestro `best_threshold` calibrado en la Tarea 1.

> **Idea clave:** si os encontráis reescribiendo todo el preprocesamiento desde cero, parad un momento — es prácticamente el mismo código que ya escribisteis en la Práctica 7, con un único valor distinto.

### Cerrando el círculo del threshold

En el Bloque 9.1 calibrasteis `best_threshold` usando el conjunto de **validación** — precisamente porque no podíais usar train (sesgado) ni test (no tenéis sus máscaras para medir Dice ahí). Ahora, en la Tarea 9.3, ese mismo valor fijo se aplica al conjunto de **test**, sin volver a recalcularlo. Este es el ciclo completo y correcto de un hiperparámetro: se calibra una vez en validación, y ese valor congelado es el que se despliega sobre datos nuevos.

> **Idea clave:** si en este punto os encontrarais recalculando el threshold sobre el propio test, estaríais repitiendo exactamente el error que el Bloque 9.1 os explicó por qué evitar.

### Un matiz técnico antes de redimensionar la máscara

El modelo predice siempre a resolución `IMG_SIZE`, pero cada imagen de test tiene su propio tamaño original — así que en algún momento tendréis que redimensionar la máscara predicha de vuelta a `(original_h, original_w)`, igual que en la Práctica 7.

Aquí hay un detalle que conviene tener claro **antes** de escribir el código, porque es un bug fácil de cometer sin que salte ningún error: si redimensionáis una máscara que **ya es binaria** (solo ceros y unos) con `cv2.resize`, la interpolación por defecto puede mezclar valores en los bordes del tumor, dejando píxeles con valores intermedios (ni 0 ni 1):

```
máscara binaria pequeña (IMG_SIZE)  →  cv2.resize  →  ya no es puramente 0/1 en los bordes
```

Tenéis dos formas válidas de evitarlo:

1. **Binarizar, redimensionar, y volver a binarizar**: aplicáis el threshold a `IMG_SIZE`, redimensionáis esa máscara, y aplicáis un segundo corte (por ejemplo `> 0.5`) después del resize para limpiar los valores intermedios que introdujo la interpolación.
2. **Redimensionar las probabilidades, binarizar una sola vez**: redimensionáis la salida continua de `sigmoid()` (no la máscara binaria) al tamaño original, y aplicáis `best_threshold` una única vez, ya en la resolución final.

Cualquiera de las dos es correcta. Lo que **no** es correcto es redimensionar una máscara binaria y pasársela directamente a `mask_to_rle` sin ese segundo corte — el RLE resultante tendría límites de tumor ligeramente distintos a los que vuestro modelo predijo realmente, sin que nada en la ejecución os avise del problema.

### Por qué un nombre de archivo distinto

La tarea pide guardar `submission_final.csv`, no sobrescribir el `submission.csv` de la Práctica 7. Mantener nombres distintos por versión os permite comparar en el Leaderboard de Kaggle qué mejora concreta aportó cada paso (baseline → augmentation → threshold óptimo), en vez de perder esa trazabilidad sobrescribiendo el mismo fichero cada vez.

### El pipeline completo, de un vistazo

```
P7: Modelo entrenado (threshold fijo 0.5)
        │
P8: ¿Mejora con augmentation?  →  modelo definitivo
        │
Bloque 9.1: best_threshold calibrado en validación
        │
Tarea 9.3: modelo definitivo + best_threshold  →  submission_final.csv
```

### Resumen rápido

| Concepto | Idea principal |
|----------|-----------------|
| Reutilización | El pipeline es conceptualmente el del Bloque 7.4; solo cambia el threshold usado |
| Ciclo del hiperparámetro | Se calibra una vez en validación, se aplica fijo sobre test |
| Binarizar vs redimensionar | Redimensionar una máscara ya binaria reintroduce valores intermedios; hay que re-binarizar o redimensionar las probabilidades antes de umbralizar |
| Nombrado de ficheros | Versionar las submissions permite atribuir la mejora a cada cambio concreto |

## Tarea 9.3: Generación de la Submission Final
Ha llegado el momento de la verdad. Vamos a aplicar todo lo aprendido (mejor modelo, mejor threshold) al conjunto de test para generar nuestro `submission.csv` definitivo.

### Instrucciones:
1. Itera sobre todas las imágenes de la carpeta `test/images`.
2. Preprocesa cada imagen exactamente igual que en la Tarea 9.2.
3. Predice la máscara usando el **`best_threshold`** que encontraste en la Tarea 9.1.
4. Redimensiona la máscara al tamaño original de la imagen.
5. Convierte la máscara a formato **RLE** (usando la función `mask_to_rle` de la P6).
6. Guarda los resultados en un DataFrame y expórtalo como **`submission_final.csv`**.

> **CHECKPOINT P9.2:** Descarga el `submission_final.csv`, súbelo a Kaggle y muestra al profesor tu nueva puntuación en el Leaderboard. ¡A por el top 10!

In [ ]:
# ============================================================
# TAREA 9.3: GENERACIÓN DE LA SUBMISSION FINAL
# ============================================================
import os
import pandas as pd

# Asegúrate de tener la función mask_to_rle definida (cópiala de la P6 si es necesario)
# def mask_to_rle(mask): ...

test_img_dir = DATA_DIR / 'test' / 'images'
test_images = sorted(list(test_img_dir.glob('*.png')))

results = []

print(f" Generando submission final con threshold = {best_threshold:.2f}...")
model.eval()

with torch.no_grad():
    for idx, img_path in enumerate(test_images):
        # 1. Cargar y preprocesar (igual que en Tarea 9.2)
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        original_h, original_w = img_rgb.shape[:2]

        # ... (Tu código de preprocesamiento aquí) ...

        # 2. Predecir con best_threshold
        # ... (Tu código de predicción aquí) ...

        # 3. Convertir a RLE y guardar
        rle = mask_to_rle(mask_binary)
        results.append({'Id': img_path.name, 'Expected': rle})

        if (idx + 1) % 50 == 0:
            print(f"   Progreso: {idx + 1}/{len(test_images)} imágenes")

# Crear DataFrame y guardar
submission_df = pd.DataFrame(results)
submission_df.to_csv('submission_final.csv', index=False)
print("submission_final.csv generado correctamente. ¡Súbelo a Kaggle!")